In [4]:
# ==============================================================================
# Step 0: Installation and Setup
# ==============================================================================
# We install `datasets` to fetch IndicCorpV2 from HuggingFace, and `nltk` for
# sentence tokenization.

!pip install datasets nltk numpy tqdm

import re
import math
import random
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

# Set seeds for reproducible results
random.seed(42)
np.random.seed(42)

# ==============================================================================
# Step 1: Data Collection & Sentence Tokenization
# ==============================================================================
# IndicCorpV2 contains paragraphs across 23 languages/scripts.
# ==============================================================================
# Step 1: Data Collection & Sentence Tokenization (FIXED SPLIT NAMES)
# ==============================================================================

from datasets import load_dataset

# Exact 23 split names shown in the IndicCorpV2 HF dataset interface
LANG_SPLITS = [
    "asm_Beng", "ben_Beng", "brx_Deva", "doi_Deva", "gom_Deva",
    "guj_Gujr", "hin_Deva", "kan_Knda", "kas_Arab", "mai_Deva",
    "mal_Mlym", "mar_Deva", "mni_Mtei", "npi_Deva", "ory_Orya",
    "pan_Guru", "san_Deva", "snd_Deva", "tam_Taml", "tel_Telu",
    "urd_Arab", "khasi", "santhali"
]

print("--- 1. Fetching IndicCorpV2 Data & Extracting Sentences ---")

raw_dataset = {}
target_sentences_per_lang = 1000

for split_name in LANG_SPLITS:
    print(f"Processing split: {split_name}...")
    try:
        # Pass the dataset name and the exact split from the repository
        ds = load_dataset(
            "ai4bharat/IndicCorpV2",
            split=split_name,
            streaming=True
        )

        sentences = []
        for record in ds:
            text = record.get("text", "")
            if not text:
                continue

            # Tokenize paragraph into sentences
            parsed_sents = sent_tokenize(text)
            for s in parsed_sents:
                s_clean = s.strip()
                if len(s_clean.split()) >= 3:
                    sentences.append(s_clean)
                if len(sentences) >= target_sentences_per_lang:
                    break
            if len(sentences) >= target_sentences_per_lang:
                break

        if len(sentences) > 0:
            raw_dataset[split_name] = sentences[:target_sentences_per_lang]
            print(f"Successfully collected {len(raw_dataset[split_name])} sentences for {split_name}.")
        else:
            print(f"Warning: No valid sentences extracted for {split_name}.")

    except Exception as e:
        print(f"Skipping or encountered error for {split_name}: {e}")

# Map language labels to integer IDs
valid_langs = sorted(list(raw_dataset.keys()))
label_to_id = {lang: i for i, lang in enumerate(valid_langs)}
id_to_label = {i: lang for i, lang in enumerate(valid_langs)}

print(f"\nCollected data across {len(valid_langs)} languages.")

# Stratified Train/Val/Test Split (80 / 10 / 10 ratio)
train_texts, train_labels = [], []
val_texts, val_labels = [], []
test_texts, test_labels = [], []

for lang in valid_langs:
    sents = raw_dataset[lang]
    random.shuffle(sents)

    n_total = len(sents)
    n_train = int(0.8 * n_total)
    n_val = int(0.1 * n_total)

    label_id = label_to_id[lang]

    train_texts.extend(sents[:n_train])
    train_labels.extend([label_id] * n_train)

    val_texts.extend(sents[n_train:n_train + n_val])
    val_labels.extend([label_id] * n_val)

    test_texts.extend(sents[n_train + n_val:])
    test_labels.extend([label_id] * (n_total - (n_train + n_val)))

print(f"Total Train Samples: {len(train_texts)}")
print(f"Total Val Samples:   {len(val_texts)}")
print(f"Total Test Samples:  {len(test_texts)}")


# ==============================================================================
# Step 2: Custom TF-IDF Vectorizer from Scratch
# ==============================================================================

class CustomTFIDFVectorizer:
    """
    Custom TF-IDF Vectorizer extracting:
    1. Word Unigrams & Bigrams
    2. Character 2-grams, 3-grams, and 4-grams
    Without using any sklearn classes.
    """
    def __init__(self, max_features=10000):
        self.max_features = max_features
        self.vocab = {}
        self.idf = {}

    def _extract_ngrams(self, text):
        features = []
        words = re.findall(r'\w+', text.lower())

        # Word Unigrams
        features.extend([f"W1_{w}" for w in words])

        # Word Bigrams
        for i in range(len(words) - 1):
            features.append(f"W2_{words[i]}_{words[i+1]}")

        # Character 2-grams, 3-grams, 4-grams
        for n in [2, 3, 4]:
            for i in range(len(text) - n + 1):
                features.append(f"C{n}_{text[i:i+n]}")

        return features

    def fit(self, raw_documents):
        print("\nBuilding vocabulary and calculating Document Frequencies (DF)...")
        df_counter = Counter()
        total_docs = len(raw_documents)

        for doc in tqdm(raw_documents):
            unique_feats = set(self._extract_ngrams(doc))
            for feat in unique_feats:
                df_counter[feat] += 1

        # Select top `max_features` based on Document Frequency
        most_common = df_counter.most_common(self.max_features)
        self.vocab = {feat: idx for idx, (feat, _) in enumerate(most_common)}

        # Calculate Smooth IDF: log((1 + N) / (1 + DF)) + 1
        print("Computing IDF values...")
        for feat, idx in self.vocab.items():
            df_val = df_counter[feat]
            self.idf[feat] = math.log((1.0 + total_docs) / (1.0 + df_val)) + 1.0

    def transform(self, raw_documents):
        num_docs = len(raw_documents)
        vocab_size = len(self.vocab)
        X = np.zeros((num_docs, vocab_size), dtype=np.float32)

        for doc_idx, doc in enumerate(tqdm(raw_documents)):
            feats = self._extract_ngrams(doc)
            feat_counts = Counter(feats)
            total_feats = len(feats)

            if total_feats == 0:
                continue

            for feat, count in feat_counts.items():
                if feat in self.vocab:
                    col_idx = self.vocab[feat]
                    # Term Frequency (TF) = count / total features in doc
                    tf = count / total_feats
                    X[doc_idx, col_idx] = tf * self.idf[feat]

            # L2 Normalization per document row
            row_norm = np.linalg.norm(X[doc_idx])
            if row_norm > 0:
                X[doc_idx] /= row_norm

        return X

    def fit_transform(self, raw_documents):
        self.fit(raw_documents)
        return self.transform(raw_documents)

# Vectorize texts
# Limit max_features to keep execution speed fast and memory efficient in Colab
vectorizer = CustomTFIDFVectorizer(max_features=12000)

print("\n--- 2. Vectorizing Training Data ---")
X_train = vectorizer.fit_transform(train_texts)

print("\n--- Vectorizing Validation Data ---")
X_val = vectorizer.transform(val_texts)

print("\n--- Vectorizing Test Data ---")
X_test = vectorizer.transform(test_texts)

y_train = np.array(train_labels, dtype=np.int32)
y_val = np.array(val_labels, dtype=np.int32)
y_test = np.array(test_labels, dtype=np.int32)

# ==============================================================================
# Step 3: Custom Logistic Regression Classifier from Scratch
# ==============================================================================

class CustomLogisticRegression:
    """
    Multinomial Logistic Regression (Softmax Classifier) using
    Mini-Batch Gradient Descent with L2 Regularization built from scratch.
    """
    def __init__(self, num_classes, learning_rate=0.1, reg_lambda=1e-4, epochs=30, batch_size=128):
        self.num_classes = num_classes
        self.lr = learning_rate
        self.reg_lambda = reg_lambda
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = None

    def _softmax(self, z):
        # Numerical stability trick subtracting max
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit(self, X, y, X_val=None, y_val=None):
        num_samples, num_features = X.shape
        self.weights = np.zeros((num_features, self.num_classes), dtype=np.float32)
        self.bias = np.zeros((1, self.num_classes), dtype=np.float32)

        # One-hot encode labels
        y_one_hot = np.zeros((num_samples, self.num_classes), dtype=np.float32)
        y_one_hot[np.arange(num_samples), y] = 1.0

        for epoch in range(1, self.epochs + 1):
            # Shuffle training dataset each epoch
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            X_shuffled = X[indices]
            y_shuffled = y_one_hot[indices]

            for start_idx in range(0, num_samples, self.batch_size):
                end_idx = min(start_idx + self.batch_size, num_samples)
                X_batch = X_shuffled[start_idx:end_idx]
                y_batch = y_shuffled[start_idx:end_idx]

                # Forward Pass
                logits = np.dot(X_batch, self.weights) + self.bias
                probs = self._softmax(logits)

                # Compute Gradients
                batch_len = X_batch.shape[0]
                dz = (probs - y_batch) / batch_len
                dw = np.dot(X_batch.T, dz) + self.reg_lambda * self.weights
                db = np.sum(dz, axis=0, keepdims=True)

                # Update Weights and Biases
                self.weights -= self.lr * dw
                self.bias -= self.lr * db

            if epoch % 5 == 0 or epoch == self.epochs:
                train_preds = self.predict(X)
                train_acc = np.mean(train_preds == y)
                val_info = ""
                if X_val is not None and y_val is not None:
                    val_preds = self.predict(X_val)
                    val_acc = np.mean(val_preds == y_val)
                    val_info = f" | Val Acc: {val_acc*100:.2f}%"
                print(f"Epoch {epoch}/{self.epochs} - Train Acc: {train_acc*100:.2f}%{val_info}")

    def predict(self, X):
        logits = np.dot(X, self.weights) + self.bias
        probs = self._softmax(logits)
        return np.argmax(probs, axis=1)

print("\n--- 3. Training Custom Logistic Regression Classifier ---")
num_classes = len(valid_langs)
clf = CustomLogisticRegression(num_classes=num_classes, learning_rate=0.5, epochs=25, batch_size=128)
clf.fit(X_train, y_train, X_val, y_val)

# ==============================================================================
# Step 4: Custom Macro-F1 Evaluation from Scratch
# ==============================================================================

def compute_macro_f1(y_true, y_pred, num_classes):
    """
    Calculates the Macro-F1 score across all classes manually without using sklearn.
    """
    f1_scores = []

    for c in range(num_classes):
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        if precision + recall > 0:
            f1 = 2 * (precision * recall) / (precision + recall)
        else:
            f1 = 0.0

        f1_scores.append(f1)

    macro_f1 = float(np.mean(f1_scores))
    return macro_f1, f1_scores

print("\n--- 4. Evaluating on Test Set ---")
y_test_pred = clf.predict(X_test)
macro_f1, per_class_f1 = compute_macro_f1(y_test, y_test_pred, num_classes)
test_accuracy = np.mean(y_test_pred == y_test)

print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Final Test Macro-F1 Score: {macro_f1:.4f}\n")

print("--- Per-Language F1 Scores ---")
for idx, f1_val in enumerate(per_class_f1):
    lang_code = id_to_label[idx]
    print(f"Language: {lang_code:<8} | F1-Score: {f1_val:.4f}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- 1. Fetching IndicCorpV2 Data & Extracting Sentences ---
Processing split: asm_Beng...
Successfully collected 1000 sentences for asm_Beng.
Processing split: ben_Beng...
Successfully collected 1000 sentences for ben_Beng.
Processing split: brx_Deva...
Successfully collected 1000 sentences for brx_Deva.
Processing split: doi_Deva...
Successfully collected 1000 sentences for doi_Deva.
Processing split: gom_Deva...
Successfully collected 1000 sentences for gom_Deva.
Processing split: guj_Gujr...
Successfully collected 1000 sentences for guj_Gujr.
Processing split: hin_Deva...
Successfully collected 1000 sentences for hin_Deva.
Processing split: kan_Knda...
Successfully collected 1000 sentences for kan_Knda.
Processing split: kas_Arab...
Successfully collected 1000 sentences for kas_Arab.
Processing split: mai_Deva...
Successfully collected 1000 sentences for mai_Deva.
Processing split: mal_Mlym...
Successfully collected 1000 sentences for mal_Mlym.
Processing split: mar_Deva...
Successf

100%|██████████| 18400/18400 [00:07<00:00, 2393.58it/s]


Computing IDF values...


100%|██████████| 18400/18400 [00:08<00:00, 2178.55it/s]



--- Vectorizing Validation Data ---


100%|██████████| 2300/2300 [00:00<00:00, 2477.20it/s]



--- Vectorizing Test Data ---


100%|██████████| 2300/2300 [00:00<00:00, 2360.73it/s]



--- 3. Training Custom Logistic Regression Classifier ---
Epoch 5/25 - Train Acc: 93.16% | Val Acc: 92.35%
Epoch 10/25 - Train Acc: 93.77% | Val Acc: 93.04%
Epoch 15/25 - Train Acc: 94.13% | Val Acc: 93.65%
Epoch 20/25 - Train Acc: 94.66% | Val Acc: 93.91%
Epoch 25/25 - Train Acc: 94.71% | Val Acc: 94.04%

--- 4. Evaluating on Test Set ---

Final Test Accuracy: 93.83%
Final Test Macro-F1 Score: 0.9369

--- Per-Language F1 Scores ---
Language: asm_Beng | F1-Score: 0.9950
Language: ben_Beng | F1-Score: 0.9950
Language: brx_Deva | F1-Score: 0.8588
Language: doi_Deva | F1-Score: 0.9005
Language: gom_Deva | F1-Score: 0.5444
Language: guj_Gujr | F1-Score: 1.0000
Language: hin_Deva | F1-Score: 0.8507
Language: kan_Knda | F1-Score: 1.0000
Language: kas_Arab | F1-Score: 0.9950
Language: khasi    | F1-Score: 0.9852
Language: mai_Deva | F1-Score: 0.8529
Language: mal_Mlym | F1-Score: 1.0000
Language: mar_Deva | F1-Score: 0.6765
Language: mni_Mtei | F1-Score: 0.9950
Language: npi_Deva | F1-Score: